# RK1828 Qwen3-Embedding 文本向量 Demo


[Qwen3-Embedding-0.6B](https://github.com/QwenLM/Qwen3-Embedding)：
把文本编码成 **1024 维向量**，用于语义搜索 / RAG 检索。GRQ W4A16 量化，
实测单条 prefill 76.7ms； paraphrase 余弦相似度 0.958 vs 无关文本 0.283。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 输入若干句子并批量编码 |
| 4 | 余弦相似度矩阵 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("qwen3-embedding-0.6b-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/qwen3_embedding/demo_x86_64.tar.gz"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.embed.bin", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.embed.bin"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.rknn", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.rknn"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.tokenizer.gguf", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.tokenizer.gguf"),
    ("qwen3-embedding-0.6b-rknn3-rk1828", "model/Qwen3-Embedding-0.6B.weight", "demos/qwen3_embedding/model/Qwen3-Embedding-0.6B.weight"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/qwen3_embedding/demo_x86_64.tar.gz", "demos/qwen3_embedding")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 输入句子并编码

每行一句。demo 每次编码一句，把 1024 维 float32 向量写到 `embedding_out.bin`。

In [ ]:
import ipywidgets as widgets
from IPython.display import display

w_texts = widgets.Textarea(
    value="How do I reset my password?\n"
          "I forgot my password, how can I change it?\n"
          "What's the best way to cook rice?\n"
          "如何重置我的密码？",
    description="句子(每行一句):", layout=widgets.Layout(width="75%", height="120px"))
display(w_texts)
print("选好后继续往下运行 ▼")

In [ ]:
import numpy as np

DEMO = "demos/qwen3_embedding"
texts = [t.strip() for t in w_texts.value.splitlines() if t.strip()]
embs = []
for t in texts:
    run_demo(["./rknn_qwen3_embedding_demo",
              "model/Qwen3-Embedding-0.6B.rknn", "model/Qwen3-Embedding-0.6B.weight",
              "model/Qwen3-Embedding-0.6B.tokenizer.gguf",
              "model/Qwen3-Embedding-0.6B.embed.bin",
              "0xff", t], DEMO)
    embs.append(np.fromfile(f"{DEMO}/embedding_out.bin", dtype=np.float32))
embs = np.stack(embs)
print("向量矩阵:", embs.shape)

## 4. 余弦相似度矩阵

语义相近的句子（含跨语言）相似度应该接近 1，无关的接近 0。

In [ ]:
import matplotlib.pyplot as plt
from vision_post import setup_cjk_font

setup_cjk_font()
norm = embs / np.linalg.norm(embs, axis=1, keepdims=True)
sim = norm @ norm.T

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(texts)), [t[:18] for t in texts], rotation=45, ha="right")
ax.set_yticks(range(len(texts)), [t[:18] for t in texts])
for i in range(len(texts)):
    for j in range(len(texts)):
        ax.text(j, i, f"{sim[i,j]:.2f}", ha="center", va="center",
                color="white" if sim[i, j] > 0.6 else "black")
plt.colorbar(im)
plt.title("余弦相似度")
plt.tight_layout()
plt.show()

## 小结

- Embedding 模型只看"语义距离"，与语种无关 —— 可以试试中英混排的同义句。
- 这是 RAG 系统的检索侧组件；配合 16 号 notebook 的 Reranker 就是完整的检索 + 精排。